# RobustBench-LLM: QLoRA Classifier
Fine-tuning Llama 3.2 1B for Security Classification

**Runtime:** Make sure you set Runtime > Change runtime type > **T4 GPU**

In [ ]:
# Install deps WITHOUT touching torch (Colab already has the right CUDA build).
# Then remove torchvision/torchaudio — they get broken by pip resolving a
# different torch wheel, and we dont need them for text classification.
!pip install -q transformers==4.46.1 datasets==3.0.1 peft==0.13.0 "bitsandbytes>=0.45.0" trl==0.12.0 accelerate==0.34.2 huggingface_hub==0.26.2 triton==3.1.0
!pip uninstall -y torchvision torchaudio 2>/dev/null || true

In [ ]:
import torch
import os, json, time, shutil
from huggingface_hub import login
from google.colab import files, userdata

# ── GPU sanity check ──
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Torch: {torch.__version__}  CUDA: {torch.version.cuda}")

# ── HuggingFace auth ──
# Set your HF token in Colab Secrets (key icon in left sidebar) as HF_TOKEN
# Or replace the line below with: login(token="your_token_here")
login(token=userdata.get("HF_TOKEN"))
print("Logged in to HuggingFace!")

In [ ]:
MODEL_ID = 'meta-llama/Llama-3.2-1B-Instruct'
MODEL_TYPE = 'causal'

if not os.path.exists('train.jsonl'):
    print('Upload train.jsonl and val.jsonl from data/processed/:')
    uploaded = files.upload()
else:
    print('Files already uploaded!')

In [ ]:
from datasets import Dataset

def load_and_format(path):
    samples = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)
            text = (
                f"System: You are a security classifier. "
                f"Answer with just 'adversarial' or 'benign'.\n"
                f"User: {item['prompt'][:800]}\n"
                f"Assistant: {item['label']}"
            )
            samples.append({'text': text})
    return Dataset.from_list(samples)

train_dataset = load_and_format('train.jsonl')
val_dataset   = load_and_format('val.jsonl')
print(f'Data loaded! Train: {len(train_dataset)}, Val: {len(val_dataset)}')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Model loaded in 4-bit precision!')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules='all-linear',
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
from trl import SFTTrainer, SFTConfig

print('Starting QLoRA training...')
start = time.time()

training_args = SFTConfig(
    output_dir='./classifier_output',
    max_seq_length=512,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,          # effective batch = 4*4 = 16
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataset_text_field='text',
    report_to='none',
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=10,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()
print(f'\nTraining completed in {(time.time()-start)/60:.1f} minutes!')

In [ ]:
model.save_pretrained('./classifier_adapter')
tokenizer.save_pretrained('./classifier_adapter')

with open('./classifier_adapter/robustbench_config.json', 'w') as f:
    json.dump({'model_id': MODEL_ID, 'model_type': MODEL_TYPE}, f)

shutil.make_archive('classifier_adapter', 'zip', './classifier_adapter')
print('Downloading adapter... place the zip contents in models/classifier/final_adapter/')
files.download('classifier_adapter.zip')